# CNN Building and Training
Here we will build our Convolutional Neural Network and train it on the FER-2013 dataset. We will configure PyTorch to use all available CPU cores to maximize performance.


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import os
import numpy as np
from pathlib import Path

# Configure PyTorch to use MAXIMUM CPU threads
num_cores = os.cpu_count()
torch.set_num_threads(num_cores)
print(f"Configured PyTorch to use all {num_cores} CPU cores for maximum performance!")

device = torch.device('cpu')

Configured PyTorch to use all 12 CPU cores for maximum performance!


## 1. Load Data
We recreate the data loaders from the previous step. We set `num_workers` to utilize multiprocessing during data loading.


In [2]:
dataset_path = Path('../dataset')
train_dir = dataset_path / 'train'
test_dir = dataset_path / 'test'

train_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

test_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = ImageFolder(root=train_dir, transform=train_transforms)
test_dataset = ImageFolder(root=test_dir, transform=test_transforms)

# Use multiple workers for data loading to keep the CPU fed
workers = min(4, num_cores) 

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=workers)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=workers)
print("Data loaders ready.")

Data loaders ready.


## 2. Define the CNN Architecture
We will build a VGG-style CNN tailored for 48x48 images. It includes Convolutional layers for feature extraction, Max Pooling for downsampling, Batch Normalization for stability, and Dropout to prevent overfitting.


In [3]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes=7):
        super(EmotionCNN, self).__init__()
        
        # Block 1: 48x48 -> 24x24
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Block 2: 24x24 -> 12x12
        self.block2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Block 3: 12x12 -> 6x6
        self.block3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        
        # Fully Connected Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

model = EmotionCNN(num_classes=7).to(device)
print("Model Architecture created!")

Model Architecture created!


## 3. Training Loop with Early Stopping
We will define our loss function, optimizer, and the training loop. We will monitor the validation loss, and save the model weights only when the validation loss improves (Model Checkpointing). We will stop early if it doesn't improve for several epochs.


In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10 # We can increase this later, starting with 10 for initial test
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0

train_losses, val_losses = [], []
train_accs, val_accs = [], []

Path('../models').mkdir(exist_ok=True)
save_path = '../models/best_emotion_cnn.pth'

print("Starting training...")
for epoch in range(num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_loader.dataset)
    epoch_train_acc = correct / total
    train_losses.append(epoch_train_loss)
    train_accs.append(epoch_train_acc)
    
    # Validation Phase
    model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_running_loss / len(test_loader.dataset)
    epoch_val_acc = val_correct / val_total
    val_losses.append(epoch_val_loss)
    val_accs.append(epoch_val_acc)
    
    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
          f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
    
    # Early Stopping & Checkpointing
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), save_path)
        print(f"--> Validation loss improved! Saved model to {save_path}")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered! No improvement for {patience} epochs.")
            break

print("Training Complete!")

Starting training...
Epoch 1/10 - Train Loss: 1.5164, Train Acc: 0.4116 | Val Loss: 1.2684, Val Acc: 0.5148
--> Validation loss improved! Saved model to ../models/best_emotion_cnn.pth
Epoch 2/10 - Train Loss: 1.2906, Train Acc: 0.5061 | Val Loss: 1.1934, Val Acc: 0.5421
--> Validation loss improved! Saved model to ../models/best_emotion_cnn.pth
Epoch 3/10 - Train Loss: 1.2232, Train Acc: 0.5340 | Val Loss: 1.1315, Val Acc: 0.5666
--> Validation loss improved! Saved model to ../models/best_emotion_cnn.pth
Epoch 4/10 - Train Loss: 1.1793, Train Acc: 0.5522 | Val Loss: 1.0875, Val Acc: 0.5830
--> Validation loss improved! Saved model to ../models/best_emotion_cnn.pth
Epoch 5/10 - Train Loss: 1.1441, Train Acc: 0.5639 | Val Loss: 1.0734, Val Acc: 0.5921
--> Validation loss improved! Saved model to ../models/best_emotion_cnn.pth
Epoch 6/10 - Train Loss: 1.1157, Train Acc: 0.5782 | Val Loss: 1.0521, Val Acc: 0.6037
--> Validation loss improved! Saved model to ../models/best_emotion_cnn.pth
E

## 4. Plot Training History
Finally, we will save the loss and accuracy metrics to file, so we can plot them in the evaluation step.


In [5]:
import pickle

history = {
    'train_loss': train_losses,
    'val_loss': val_losses,
    'train_acc': train_accs,
    'val_acc': val_accs
}

with open('../models/training_history.pkl', 'wb') as f:
    pickle.dump(history, f)

print("Saved training history to models/training_history.pkl")

Saved training history to models/training_history.pkl
